# Two-Stage Hybrid Recommender Experiment

This notebook follows the WideDeep experiment style and evaluates the new
`TwoStageHybridRecommender` on both split protocols:
1. Global temporal split (`train/val/test`)
2. Per-user temporal split (`user_based_temporal_train/val`)


## 1. Imports and Data Loading


In [1]:
!git clone https://$GITHUB_TOKEN@github.com/Its-OP/ucu-rs-2026.git

Cloning into 'ucu-rs-2026'...
remote: Enumerating objects: 625, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 625 (delta 87), reused 80 (delta 73), pack-reused 521 (from 1)
Receiving objects: 100% (625/625), 80.22 MiB | 6.24 MiB/s, done.
Resolving deltas: 100% (312/312), done.


In [2]:
%cd ucu-rs-2026

/Users/anastasiiamazur/Projects/ucu-rs-2026/experiments/HybridTwoStage/ucu-rs-2026


In [3]:
import os
import re
import shlex
import subprocess
import sys
import time
from dataclasses import dataclass

import pandas as pd

from src import run_hybrid_two_stage as run_hybrid


## 2. Helpers


In [4]:
@dataclass
class RunResult:
    name: str
    split_type: str
    split: str
    evaluator: str
    command: str
    elapsed_seconds: float
    ndcg_at_10: float | None
    precision_at_10: float | None
    recall_at_10: float | None
    mrr_at_10: float | None
    map_at_10: float | None
    n_users_evaluated: int | None
    skip_rate: float | None
    return_code: int
    success: bool
    error_kind: str | None
    raw_output: str


def _extract_float(pattern: str, text: str):
    m = re.search(pattern, text)
    return float(m.group(1)) if m else None


def _extract_int(pattern: str, text: str):
    m = re.search(pattern, text)
    return int(m.group(1)) if m else None


def parse_metrics(stdout: str):
    ndcg = _extract_float(r"k=10:\s+ndcg=([0-9.]+)", stdout)
    precision = _extract_float(r"k=10:[^\n]*precision=([0-9.]+)", stdout)
    recall = _extract_float(r"k=10:[^\n]*recall=([0-9.]+)", stdout)
    mrr = _extract_float(r"k=10:[^\n]*mrr=([0-9.]+)", stdout)
    map_k10 = _extract_float(r"k=10:[^\n]*map=([0-9.]+)", stdout)

    if ndcg is None:
        ndcg = _extract_float(r"NDCG@10:\s+([0-9.]+)", stdout)
        precision = _extract_float(r"Precision@10:\s+([0-9.]+)", stdout)
        recall = _extract_float(r"Recall@10:\s+([0-9.]+)", stdout)

    n_eval = _extract_int(r"evaluated=([0-9]+)", stdout)
    skip_rate = _extract_float(r"skip rate=([0-9.]+)", stdout)
    return ndcg, precision, recall, mrr, map_k10, n_eval, skip_rate


def _error_kind(return_code: int, output: str) -> str | None:
    if return_code == 0:
        return None
    if return_code == -11:
        return "segmentation_fault"
    if "ModuleNotFoundError" in output:
        return "missing_dependency"
    if "MemoryError" in output:
        return "memory_error"
    return "runtime_error"


def run_hybrid(
    name: str,
    split_type: str,
    split: str,
    evaluator: str,
    extra_args: str = "",
    strict: bool = False,
) -> RunResult:
    cmd = (
        f"{shlex.quote(sys.executable)} -m src.run_hybrid_two_stage "
        f"--split-type {split_type} --split {split} "
        f"--evaluator {evaluator} --ks 10,20 --k 10 "
        f"{extra_args}"
    ).strip()
    print(f"\n=== {name} ===")
    print(cmd)

    safe_env = os.environ.copy()
    safe_env.update({
        "PYTHONFAULTHANDLER": "1",
        "OMP_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
        "VECLIB_MAXIMUM_THREADS": "1",
        "NUMEXPR_NUM_THREADS": "1",
    })

    t0 = time.perf_counter()
    proc = subprocess.run(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
        env=safe_env,
    )
    elapsed = time.perf_counter() - t0

    out = proc.stdout
    print(out)

    ndcg, precision, recall, mrr, map_k10, n_eval, skip_rate = parse_metrics(out)
    result = RunResult(
        name=name,
        split_type=split_type,
        split=split,
        evaluator=evaluator,
        command=cmd,
        elapsed_seconds=elapsed,
        ndcg_at_10=ndcg,
        precision_at_10=precision,
        recall_at_10=recall,
        mrr_at_10=mrr,
        map_at_10=map_k10,
        n_users_evaluated=n_eval,
        skip_rate=skip_rate,
        return_code=int(proc.returncode),
        success=(proc.returncode == 0),
        error_kind=_error_kind(proc.returncode, out),
        raw_output=out,
    )

    if strict and not result.success:
        raise RuntimeError(f"Command failed with code {proc.returncode}: {cmd}")

    return result


## 3. Global Split Experiments

Tune on `global/val`, then run final report on `global/test` using the best config by NDCG@10.


In [5]:

# Top-5 representative configurations covering the design space explored so far.

# Known test results (global split):
#   blend=1.0  → NDCG@10=0.23863, MRR@10=0.37856
#   blend=0.8  → NDCG@10=0.24517, MRR@10=0.38583
#   blend=0.6  → NDCG@10=0.24762, MRR@10=0.39331  ← current leader
#   blend=0.4  → untested on test set
_COMMON = (
    "--mode all "
    "--content-n-neighbors 5 "
    "--cf-candidates 400 --cb-candidates 120 "
    "--train-cf-candidates 200 --train-cb-candidates 60 "
    "--rerank-holdout-frac 0.10 "
    "--seed 42"
)

global_variants = [
    {
        "name": "hybrid_no_ranker_rrf085",
        "extra_args": f"{_COMMON} --disable-ranker --blend-alpha 0.85",
        "note": "Ablation: CF-dominant RRF, no ML reranker",
    },
    {
        "name": "hybrid_lambdamart_blend1.0",
        "extra_args": f"{_COMMON} --ranker-cf-blend 1.0",
        "note": "Pure LambdaMART output; no BPR score pull",
    },
    {
        "name": "hybrid_lambdamart_blend0.8",
        "extra_args": f"{_COMMON} --ranker-cf-blend 0.8",
        "note": "80% LambdaMART + 20% BPR CF",
    },
    {
        "name": "hybrid_lambdamart_blend0.6",
        "extra_args": f"{_COMMON} --ranker-cf-blend 0.6",
        "note": "60% LambdaMART + 40% BPR CF — best known on test",
    },
    {
        "name": "hybrid_lambdamart_blend0.4",
        "extra_args": f"{_COMMON} --ranker-cf-blend 0.4",
        "note": "40% LambdaMART + 60% BPR CF — extends blend sweep",
    },
]

global_val_runs = []
for cfg in global_variants:
    rr = run_hybrid(
        name=cfg["name"],
        split_type="global",
        split="val",
        evaluator="offline",
        extra_args=cfg["extra_args"],
        strict=False,
    )
    global_val_runs.append(rr)

global_val_df = pd.DataFrame([
    {**r.__dict__, "note": global_variants[i]["note"]}
    for i, r in enumerate(global_val_runs)
])
global_val_df[
    ["name", "note", "success", "ndcg_at_10", "precision_at_10",
     "recall_at_10", "mrr_at_10", "map_at_10",
     "n_users_evaluated", "skip_rate", "elapsed_seconds"]
].sort_values(["success", "ndcg_at_10"], ascending=[False, False])


=== hybrid_no_ranker_rrf085 ===
/Users/anastasiiamazur/Projects/ucu-rs-2026/.venv/bin/python -m src.run_hybrid_two_stage --split-type global --split val --evaluator offline --ks 10,20 --k 10 --mode all --content-n-neighbors 5 --cf-candidates 400 --cb-candidates 120 --train-cf-candidates 200 --train-cb-candidates 60 --rerank-holdout-frac 0.10 --seed 42 --disable-ranker --blend-alpha 0.85
2026-02-27 22:17:58,513 INFO __main__: Fitting TwoStageHybridRecommender...
2026-02-27 22:17:58,928 INFO src.models.hybrid_two_stage: Fitting BPR component...
2026-02-27 22:18:20,722 INFO src.models.bpr: BPR epoch 1/20, avg loss: 0.441048
2026-02-27 22:18:40,579 INFO src.models.bpr: BPR epoch 2/20, avg loss: 0.356003
2026-02-27 22:19:20,464 INFO src.models.bpr: BPR epoch 4/20, avg loss: 0.333132
2026-02-27 22:19:58,959 INFO src.models.bpr: BPR epoch 6/20, avg loss: 0.326196
2026-02-27 22:20:37,438 INFO src.models.bpr: BPR epoch 8/20, avg loss: 0.318881
2026-02-27 22:21:17,816 INFO src.models.bpr: BPR e

,name,note,success,ndcg_at_10,precision_at_10,recall_at_10,mrr_at_10,map_at_10,n_users_evaluated,skip_rate,elapsed_seconds
0,hybrid_no_ranker_rrf085,"Ablation: CF-dominant RRF, no ML reranker",True,0.29570,0.26396,0.05564,0.46636,0.17936,1188,0.80331,432.624485
4,hybrid_lambdamart_blend0.4,40% LambdaMART + 60% BPR CF — extends blend sweep,True,0.28204,0.25274,0.05213,0.44569,0.16399,1188,0.80331,423.551205
3,hybrid_lambdamart_blend0.6,60% LambdaMART + 40% BPR CF — best known on test,True,0.26607,0.24485,0.05148,0.39314,0.15521,1188,0.80331,414.968163
2,hybrid_lambdamart_blend0.8,80% LambdaMART + 20% BPR CF,True,0.24923,0.22576,0.04741,0.38380,0.13520,1188,0.80331,418.608836
1,hybrid_lambdamart_blend1.0,Pure LambdaMART output; no BPR score pull,True,0.19342,0.18288,0.03485,0.27001,0.09764,1188,0.80331,425.889621


In [6]:

# pick the best config by NDCG@10 on val, then run it on test.
global_ok_df = global_val_df[
    (global_val_df["success"]) & (global_val_df["ndcg_at_10"].notna())
]
best_global = global_ok_df.sort_values("ndcg_at_10", ascending=False).iloc[0]
print(f"Best val variant: {best_global['name']}  (NDCG@10={best_global['ndcg_at_10']:.5f})")

best_cfg = next(cfg for cfg in global_variants if cfg["name"] == best_global["name"])
best_global_test = run_hybrid(
    name=f"{best_global['name']}_test",
    split_type="global",
    split="test",
    evaluator="offline",
    extra_args=best_cfg["extra_args"],
    strict=False,
)

pd.DataFrame([{**best_global_test.__dict__, "note": best_cfg["note"]}])[
    ["name", "note", "success", "return_code", "split_type", "split",
     "ndcg_at_10", "precision_at_10", "recall_at_10", "mrr_at_10",
     "map_at_10", "n_users_evaluated", "skip_rate", "elapsed_seconds"]
]


Best val variant: hybrid_no_ranker_rrf085  (NDCG@10=0.29570)

=== hybrid_no_ranker_rrf085_test ===
/Users/anastasiiamazur/Projects/ucu-rs-2026/.venv/bin/python -m src.run_hybrid_two_stage --split-type global --split test --evaluator offline --ks 10,20 --k 10 --mode all --content-n-neighbors 5 --cf-candidates 400 --cb-candidates 120 --train-cf-candidates 200 --train-cb-candidates 60 --rerank-holdout-frac 0.10 --seed 42 --disable-ranker --blend-alpha 0.85
2026-02-27 22:53:14,357 INFO __main__: Fitting TwoStageHybridRecommender...
2026-02-27 22:53:14,773 INFO src.models.hybrid_two_stage: Fitting BPR component...
2026-02-27 22:53:34,475 INFO src.models.bpr: BPR epoch 1/20, avg loss: 0.441048
2026-02-27 22:53:53,574 INFO src.models.bpr: BPR epoch 2/20, avg loss: 0.356003
2026-02-27 22:54:32,521 INFO src.models.bpr: BPR epoch 4/20, avg loss: 0.333132
2026-02-27 22:55:11,359 INFO src.models.bpr: BPR epoch 6/20, avg loss: 0.326196
2026-02-27 22:55:51,451 INFO src.models.bpr: BPR epoch 8/20, av

,name,note,success,return_code,split_type,split,ndcg_at_10,precision_at_10,recall_at_10,mrr_at_10,map_at_10,n_users_evaluated,skip_rate,elapsed_seconds
0,hybrid_no_ranker_rrf085_test,"Ablation: CF-dominant RRF, no ML reranker",True,0,global,test,0.23709,0.21007,0.04958,0.37669,0.13406,1347,0.77699,425.246775


## 4. Per-User Split Experiments

Run the same variant family on `per_user/val`.


In [7]:

# run all 5 configs on per-user temporal split
per_user_runs = []
for cfg in global_variants:
    rr = run_hybrid(
        name=cfg["name"],
        split_type="per_user",
        split="val",
        evaluator="offline",
        extra_args=cfg["extra_args"],
        strict=False,
    )
    per_user_runs.append(rr)

per_user_df = pd.DataFrame([
    {**r.__dict__, "note": global_variants[i]["note"]}
    for i, r in enumerate(per_user_runs)
])
per_user_df[
    ["name", "note", "success", "ndcg_at_10", "precision_at_10",
     "recall_at_10", "mrr_at_10", "map_at_10",
     "n_users_evaluated", "skip_rate", "elapsed_seconds"]
].sort_values(["success", "ndcg_at_10"], ascending=[False, False])



=== hybrid_no_ranker_rrf085 ===
/Users/anastasiiamazur/Projects/ucu-rs-2026/.venv/bin/python -m src.run_hybrid_two_stage --split-type per_user --split val --evaluator offline --ks 10,20 --k 10 --mode all --content-n-neighbors 5 --cf-candidates 400 --cb-candidates 120 --train-cf-candidates 200 --train-cb-candidates 60 --rerank-holdout-frac 0.10 --seed 42 --disable-ranker --blend-alpha 0.85
2026-02-27 23:00:19,006 INFO __main__: Fitting TwoStageHybridRecommender...
2026-02-27 23:00:19,310 INFO src.models.hybrid_two_stage: Fitting BPR component...
2026-02-27 23:00:39,344 INFO src.models.bpr: BPR epoch 1/20, avg loss: 0.434580
2026-02-27 23:00:58,902 INFO src.models.bpr: BPR epoch 2/20, avg loss: 0.350822
2026-02-27 23:01:38,591 INFO src.models.bpr: BPR epoch 4/20, avg loss: 0.328740
2026-02-27 23:02:18,597 INFO src.models.bpr: BPR epoch 6/20, avg loss: 0.321364
2026-02-27 23:02:56,547 INFO src.models.bpr: BPR epoch 8/20, avg loss: 0.314320
2026-02-27 23:03:34,982 INFO src.models.bpr: BPR

,name,note,success,ndcg_at_10,precision_at_10,recall_at_10,mrr_at_10,map_at_10,n_users_evaluated,skip_rate,elapsed_seconds
0,hybrid_no_ranker_rrf085,"Ablation: CF-dominant RRF, no ML reranker",True,0.14293,0.12292,0.05751,0.26528,0.06885,5999,0.00679,424.408643
4,hybrid_lambdamart_blend0.4,40% LambdaMART + 60% BPR CF — extends blend sweep,True,0.12698,0.10900,0.05034,0.23632,0.05730,5999,0.00679,415.366992
3,hybrid_lambdamart_blend0.6,60% LambdaMART + 40% BPR CF — best known on test,True,0.12510,0.10774,0.05010,0.23049,0.05490,5999,0.00679,414.528635
2,hybrid_lambdamart_blend0.8,80% LambdaMART + 20% BPR CF,True,0.11981,0.10291,0.04894,0.21283,0.04905,5999,0.00679,416.905259
1,hybrid_lambdamart_blend1.0,Pure LambdaMART output; no BPR score pull,True,0.10534,0.08841,0.04392,0.18787,0.03934,5999,0.00679,410.590082


In [8]:

# run the best-known config (blend=0.6) explicitly on per-user split
_BEST_CFG = next(cfg for cfg in global_variants if cfg["name"] == "hybrid_lambdamart_blend0.6")
best_per_user = run_hybrid(
    name="hybrid_lambdamart_blend0.6_per_user",
    split_type="per_user",
    split="val",
    evaluator="offline",
    extra_args=_BEST_CFG["extra_args"],
    strict=False,
)
pd.DataFrame([{**best_per_user.__dict__, "note": _BEST_CFG["note"]}])[
    ["name", "note", "success", "split_type", "split",
     "ndcg_at_10", "precision_at_10", "recall_at_10", "mrr_at_10",
     "map_at_10", "n_users_evaluated", "skip_rate", "elapsed_seconds"]
]



=== hybrid_lambdamart_blend0.6_per_user ===
/Users/anastasiiamazur/Projects/ucu-rs-2026/.venv/bin/python -m src.run_hybrid_two_stage --split-type per_user --split val --evaluator offline --ks 10,20 --k 10 --mode all --content-n-neighbors 5 --cf-candidates 400 --cb-candidates 120 --train-cf-candidates 200 --train-cb-candidates 60 --rerank-holdout-frac 0.10 --seed 42 --ranker-cf-blend 0.6
2026-02-27 23:35:01,665 INFO __main__: Reranker holdout generated from training window: base_train=672936 rerank_labels=74973
2026-02-27 23:35:01,666 INFO __main__: Fitting TwoStageHybridRecommender...
2026-02-27 23:35:01,949 INFO src.models.hybrid_two_stage: Fitting BPR component...
2026-02-27 23:35:22,037 INFO src.models.bpr: BPR epoch 1/20, avg loss: 0.438777
2026-02-27 23:35:38,408 INFO src.models.bpr: BPR epoch 2/20, avg loss: 0.351692
2026-02-27 23:36:10,899 INFO src.models.bpr: BPR epoch 4/20, avg loss: 0.328003
2026-02-27 23:36:44,543 INFO src.models.bpr: BPR epoch 6/20, avg loss: 0.320536
2026

,name,note,success,split_type,split,ndcg_at_10,precision_at_10,recall_at_10,mrr_at_10,map_at_10,n_users_evaluated,skip_rate,elapsed_seconds
0,hybrid_lambdamart_blend0.6_per_user,60% LambdaMART + 40% BPR CF — best known on test,True,per_user,val,0.12576,0.10791,0.04998,0.2288,0.05514,5999,0.00679,404.082423


## 5. Global vs Per-User Comparison


In [9]:

global_ok_df  = global_val_df[(global_val_df["success"]) & (global_val_df["ndcg_at_10"].notna())]
per_user_ok_df = per_user_df[(per_user_df["success"]) & (per_user_df["ndcg_at_10"].notna())]

if global_ok_df.empty or per_user_ok_df.empty:
    raise RuntimeError("Cannot build comparison: at least one protocol has no successful run.")

best_global_val  = global_ok_df.sort_values("ndcg_at_10", ascending=False).iloc[0]
best_per_user_val = per_user_ok_df.sort_values("ndcg_at_10", ascending=False).iloc[0]

_cols = ["ndcg_at_10", "precision_at_10", "recall_at_10", "mrr_at_10",
         "map_at_10", "n_users_evaluated", "skip_rate"]

rows = [
    {"protocol": "global_temporal_val",  "variant": best_global_val["name"],
     **{c: best_global_val[c]  for c in _cols}},
    {"protocol": "global_temporal_test", "variant": best_global_test.name,
     **{"ndcg_at_10": best_global_test.ndcg_at_10,
        "precision_at_10": best_global_test.precision_at_10,
        "recall_at_10": best_global_test.recall_at_10,
        "mrr_at_10": best_global_test.mrr_at_10,
        "map_at_10": best_global_test.map_at_10,
        "n_users_evaluated": best_global_test.n_users_evaluated,
        "skip_rate": best_global_test.skip_rate}},
    {"protocol": "per_user_temporal_val", "variant": best_per_user_val["name"],
     **{c: best_per_user_val[c] for c in _cols}},
]

protocol_compare = pd.DataFrame(rows)
protocol_compare


,protocol,variant,ndcg_at_10,precision_at_10,recall_at_10,mrr_at_10,map_at_10,n_users_evaluated,skip_rate
0,global_temporal_val,hybrid_no_ranker_rrf085,0.29570,0.26396,0.05564,0.46636,0.17936,1188,0.80331
1,global_temporal_test,hybrid_no_ranker_rrf085_test,0.23709,0.21007,0.04958,0.37669,0.13406,1347,0.77699
2,per_user_temporal_val,hybrid_no_ranker_rrf085,0.14293,0.12292,0.05751,0.26528,0.06885,5999,0.00679


In [10]:
global_val_df.to_csv("global_val_runs.csv", index=False)
per_user_df.to_csv("per_user_val_runs.csv", index=False)
protocol_compare.to_csv("protocol_compare.csv", index=False)